# Loading any dataset uniformly

The [dataset documentation](../datasets/) describes source files, response
measures, acquisition and loader options in detail. Use this notebook for the
shared loading API and the [segmentation tutorial](segmentation.ipynb) for
unit alignment and recalculation from finer-grained source data.

REPLICA exposes its reading-time and EEG corpora through the same
interface. Source units, available measures and fixation support still differ. One
corpus can map to several registry entries: MECO, for example, is a
single multilingual corpus split per language and per granularity, so the
registry-entry count is larger than the number of distinct corpora. Every
entry obeys the same interface:

```python
ds = LoaderClass(...)        # constructor
ds.load(path)                # load from a local directory
ds.stimuli()                 # list[Stimulus]
ds.feature(quantity)         # Feature selecting a stored quantity
ds.measures                  # available stored quantity names
ds.default_measure           # the loader's default quantity
```

`EvaluationSpec` assigns features to targets, predictors and controls.
`evaluate(ds, spec, AnalysisConfig(...))` handles unit-aligned quantities;
recording-based EEG/fMRI targets use `EncodingConfig`.

For loader development and source inspection, `ds.data(measure=...)` returns
underlying unit-aligned `Datum` records. `ds.feature(...)` exposes those values
through the common feature interface.

## Prerequisites

```bash
pip install -e ".[models,readers,r]"
replica fetch provo natural_stories meco potec
```

`readers` installs pyreadr for the MECO release files, and `models` installs torch and transformers for the language model in section 3.

The default mixed model needs R with `lme4`; see [installation](getting_started.ipynb).


In [ ]:
import logging

from replica.dataset import available_datasets, describe_dataset, load_dataset
from replica.dataset.catalog.registry import GROUPS, expand

logging.getLogger("replica").setLevel(logging.WARNING)
print(f"{len(available_datasets())} catalog keys; groups: {sorted(GROUPS)}")

## 1. Discover datasets and their options

`available_datasets()` lists stable loadable keys. `describe_dataset()`
shows constructor options, load-time options, defaults, valid choices,
and canonical paths without loading any data. Use `expand()` when you
want to resolve a registry group into its concrete members.


In [ ]:
# Enumerate keys and inspect configurable dataset families.
print("Single name :", expand("provo"))
print("Group alias :", expand("meco_l1_w1_all")[:3], "...")
print(
    "Whole REPLICA   :",
    available_datasets()[:5],
    "... (",
    len(available_datasets()),
    "total )",
)
print()
print(describe_dataset("meco"))

## 2. Load any dataset by name

`load_dataset()` constructs the loader, separates constructor options
from load-time options, calls `.load()`, and returns a populated `Dataset`.
It reads from `$PSYCHO_DATA_DIR`, or from `datasets/` in a source checkout
when that variable is unset; `path=` reads one dataset from another directory.


In [ ]:
for name in ["provo", "natural_stories", "meco_en", "potec"]:
    ds = load_dataset(name)
    print(
        f"{name:18s}  stim={len(ds.stimuli()):>4}  "
        f"quantities={ds.measures}  default={ds.default_measure!r}"
    )

### Choose response policies explicitly

For eye-tracking loaders, `load_dataset("provo", drop_skips=False)` includes known zero-duration skips. The default excludes them; unavailable source values remain missing in either view. A later log transform will also exclude zeros.

`ds.data(measure, source="released")` selects retained released responses, with the loader's duration conversion. MECO's default first run and CopCo's default go-past require matching raw fixation reports because they use standard fixation-derived calculations. Reading their retained released summaries does not require those reports.

Use `ds.feature(measure, source="released")` to select that source for any evaluation role. `fixation_feature(ds, measure, calculation=FixationCalculation(...))` requests a calculation from validated events. Support for summaries does not imply support for raw events or arbitrary finer units. The [segmentation tutorial](segmentation.ipynb) lists supported configurations and unresolved source limitations.

## 3. Evaluate loaded responses

Select each loader's available quantities as features and assign them to evaluation roles. These examples run on the complete PROVO and Natural Stories datasets and require a language model.

In [ ]:
from replica import (
    AnalysisConfig,
    EvaluationSpec,
    ModelFeature,
    Surprisal,
    UnitLength,
    evaluate,
)
from replica.feature import HuggingFaceLM

lm = HuggingFaceLM("gpt2", add_bos=True)


def spec_template(ds, dv):
    return EvaluationSpec(
        predictors=[ModelFeature(lm, Surprisal())],
        controls=[UnitLength()],
        targets=[ds.feature(dv)],
    )


cfg = AnalysisConfig(n_folds=2)  # Two folds per axis for this short example.

for name in ["provo", "natural_stories"]:
    ds = load_dataset(name)

    dv = ds.default_measure
    results = evaluate(ds, spec_template(ds, dv), cfg)
    r = results[0]
    print(
        f"{name:18s}  {dv:18s}  ΔLL={r.delta_ll:+.2f}  "
        f"ΔR²={r.delta_r2:+.4f}  n={r.n_observations}"
    )

## 4. Looping the registry

`benchmark()` (or a plain `for` loop) over registry groups runs a single specification across many corpora at once.
MECO languages, waves, and cohorts are first-class catalog entries, so you do not
need a separate code path for them. The registry spans several modalities, so
match the selected quantities to the corpora you loop over: a reading-time measure such as
`first_run_in_first_pass` is available in many eye-tracking corpora and `spr_rt` in self-paced-reading corpora. EEG sources may expose signed ERP amplitudes, frequency-band power or recordings. Inspect `ds.measures` and the loader documentation before selecting responses; modality alone does not establish a measure.


In [ ]:
# L1W1 languages, character views, and the broader word-level registry.
print("meco_l1_w1_all    :", len(GROUPS["meco_l1_w1_all"]))
print("meco_char_all     :", len(GROUPS["meco_char_all"]))
print("all_word_level    :", len(GROUPS["all_word_level"]))
print("all (everything)  :", len(GROUPS["all"]))

## 5. When the data isn't on disk yet

If the data is not on disk, `load_dataset(...)` raises a `FileNotFoundError`.
`replica datasets` shows each dataset's status, and `replica fetch NAME`
downloads a dataset. The keys `celer`, `dundee_official`, `subtlex_en`,
`narratives` and `natural_stories_maze` need manual acquisition; for these,
`replica fetch NAME` prints the access steps.

```bash
replica datasets          # status of every dataset
replica fetch provo       # fetch one
replica fetch meco_char   # fetch the per-character export
replica fetch celer       # print the access steps for a manual source
```
